# Module 2.3 - Document Processing and Chunking

**Reference:** [`03-document-processing-and-chunking.md`](./03-document-processing-and-chunking.md)

## What you'll do in this notebook

- Implement fixed-size and recursive chunking from scratch (no LangChain required).
- Add overlap between chunks so information on boundaries isn't lost.
- Enrich each chunk with metadata (source, chunk index) for later citation.
- Build a complete process-document pipeline you can reuse in the capstone.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
print(f"OK: client ready, embedding model = {EMBED_MODEL}")

## Sample document

We'll use a short synthetic document throughout this notebook. Treat it as a stand-in for a real PDF or Markdown file you'd process in production.

In [ ]:
SAMPLE_DOC = """DevBrew Cafe - Operations Handbook

Opening hours. The cafe is open Monday to Friday from 06:00 to 22:00,
and on weekends from 08:00 to 20:00. Public holiday hours are announced
on the website the week before.

Payments. We accept Visa, Mastercard, American Express, and Apple Pay.
Cash payments are accepted up to 50 pounds per transaction.

Refund policy. Drinks may be remade or refunded within 15 minutes of
purchase. Packaged food is non-refundable once opened. Merchandise may
be returned unopened within 14 days with the original receipt.

Allergens. The kitchen handles nuts, gluten, dairy, and soy. Cross-
contamination is possible even for items labelled 'free from', so
customers with severe allergies should speak to a manager before ordering.

Loyalty programme. The Debug Stamp card gives customers one free drink
after every ten purchased. Stamps do not expire. Lost cards cannot be
replaced with stamps transferred, so we recommend registering the card
in the app.
"""

print(f"Document length: {len(SAMPLE_DOC)} characters")

## Exercise 1 - Fixed-size chunking

Implement the simplest possible chunker: walk through the string in fixed-size steps, allowing overlap between adjacent chunks.

In [ ]:
def fixed_size_chunks(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    # TODO: return a list of substrings, each at most `chunk_size` characters,
    # sliding forward by (chunk_size - overlap) characters each time.
    raise NotImplementedError

chunks = fixed_size_chunks(SAMPLE_DOC, chunk_size=300, overlap=50)
print(f"{len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\n--- chunk {i} ({len(chunk)} chars) ---\n{chunk}")

Fixed-size chunks are easy to reason about but often cut mid-sentence. Scan the output and find at least one chunk that starts or ends in the middle of a useful idea.

## Exercise 2 - Recursive splitting

A smarter chunker tries natural boundaries first - paragraphs, then sentences, then words. Here is a skeleton. Fill in the logic.

**Algorithm:** for each candidate separator in priority order, split the text. If any resulting piece is still too large, recurse on it with the next separator. If none is small enough and we've run out of separators, fall back to fixed-size chunking.

In [ ]:
SEPARATORS = ["\n\n", "\n", ". ", " "]

def recursive_split(text: str, max_chars: int = 300, separators: list[str] = SEPARATORS) -> list[str]:
    # TODO:
    # 1. If len(text) <= max_chars, return [text].
    # 2. If separators is non-empty, split on separators[0]. For each piece,
    #    if it's already small enough keep it, otherwise recurse on it with
    #    separators[1:].
    # 3. If separators is empty, fall back to fixed_size_chunks(text, max_chars, overlap=0).
    raise NotImplementedError

rec_chunks = recursive_split(SAMPLE_DOC, max_chars=300)
print(f"{len(rec_chunks)} chunks")
for i, chunk in enumerate(rec_chunks):
    print(f"\n--- chunk {i} ({len(chunk)} chars) ---\n{chunk}")

Compare your recursive output to the fixed-size output. Do the chunks feel more like coherent units of information?

## Exercise 3 - Chunk + metadata + embed pipeline

A chunk on its own is not enough for RAG - you also need metadata (so you can cite the source) and the embedding itself (so you can search). Build the full pipeline.

In [ ]:
def process_document(text: str, source: str, max_chars: int = 400) -> list[dict]:
    """Return a list of dicts ready for a vector store.

    Each dict has: id, text, embedding, metadata={source, chunk_index}.
    """
    # TODO:
    # 1. Split text with recursive_split(text, max_chars).
    # 2. Call client.embeddings.create(model=EMBED_MODEL, input=chunks)  once
    #    for all chunks (batching is free and much faster).
    # 3. Zip chunks with their embeddings and build the dicts.
    raise NotImplementedError

processed = process_document(SAMPLE_DOC, source="devbrew-handbook.md")
print(f"{len(processed)} chunks processed")
for p in processed[:2]:
    print(f"\nid: {p['id']}")
    print(f"source: {p['metadata']['source']}")
    print(f"text: {p['text'][:80]}...")
    print(f"embedding dims: {len(p['embedding'])}")

**Experiment:** re-run `process_document` with `max_chars=150` and again with `max_chars=800`. How does the chunk count change? Which size produces chunks that feel most like self-contained answers to "What are the opening hours?" or "Can I get a refund?"

## What's next

We now have a list of chunks, each with an embedding. In `04-vector-databases.ipynb` we drop them into ChromaDB and let it handle fast similarity search at query time.